In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
#@title Create Data Function
def create_data()->tuple:
    X = np.random.normal(170, 9, 600).round(1)
    y = .9*X - 88 + np.random.randn(600)*5
    sns.scatterplot(x=X, y=y, color='grey', s=60, alpha=.8, edgecolor='black').set_title('Train Data')
    plt.grid()
    plt.xlabel('Human height')
    plt.ylabel('Human weight')
    plt.show()
    return X, y

# TASK

Вы работаете с реальным экспериментальным набором данных. Мы собрали информацию о **600 студентах** в возрасте **от 18 до 25 лет**. Для каждого участника был измерен **рост** (в сантиметрах) и **масса тела** (в килограммах). Это данные взрослых людей без выраженных нарушений телосложения.


In [ ]:
X, y = create_data()

In [ ]:
X.shape

In [ ]:
y.shape

#  Цель

На основе имеющихся данных необходимо построить **линейную модель** (линейная регрессия), описывающую зависимость массы тела от роста. Иными словами, вы должны аппроксимировать зависимость вида:

$$
\hat{y} = \omega_0 + \omega_1 \cdot x
$$

где:

* $x$ — рост человека (в см),
* $\hat{y}$ — предсказанная масса тела,
* $\omega_0$, $\omega_1$ — коэффициенты, которые нужно определить.

---

##  Постановка задачи

Найдите такие значения параметров $\omega_0$, $\omega_1$, которые минимизируют функцию ошибки (лосс-функцию) — **среднеквадратичную ошибку** (MSE):

$$
L(\omega_0, \omega_1) = \frac{1}{n} \sum_{i=1}^n \left(y_i - \hat{y}_i\right)^2 = \frac{1}{n} \sum_{i=1}^n \left(y_i - \omega_0 - \omega_1 x_i \right )^2
$$

где:

* $x_i$ — рост $i$-го участника,
* $y_i$ — реальная масса $i$-го участника,
* $\hat{y}_i$ — предсказанная моделью масса,
* $n = 600$ — общее число наблюдений.

---

##  Что требуется

1. Реализуйте на Python код, который:

   * принимает на вход массивы `X` (рост) и `y` (масса),
   * численно находит $\omega_0$ и $\omega_1$, при которых MSE принимает наименьшее значение из возможных.
2. Постройте график:

   * точки реальных наблюдений,
   * предсказанную линейную зависимость.
3. Проверьте, насколько хорошо линейная модель описывает данные.

---

Для решения данной задачи не очень хорошо работает метод приближенного вычисления производных. Вам необходимо будет получить частные производные loss руками для подсчета значений градиента. В остальном, используйте метод градиентного спуска.


$$
\frac{\partial L(w)}{\partial w_1} = -\frac{2}{n} \sum_{i=1}^{n} x_i \left( y_i - w_0 - w_1 x_i \right)
$$

$$
\frac{\partial L(w)}{\partial w_0} = -\frac{2}{n} \sum_{i=1}^{n} \left( y_i - w_0 - w_1 x_i \right)
$$

In [ ]:
def predict(X: np.ndarray,  w_0: float, w_1: float)->np.ndarray:
    y_pred = X * w_1 + w_0
    return y_pred

def mse_loss(y_true: np.ndarray, y_pred: np.ndarray)-> float:
    return ((y_true - y_pred)**2).mean().item()

def dl_dw_1(X: np.ndarray, y: np.ndarray, w_0: float, w_1: float)->float:
    y_pred = predict(X, w_0, w_1)
    return (-2*(X * (y-y_pred)).mean()).item()

def dl_dw_0(X: np.ndarray, y: np.ndarray, w_0: float, w_1: float)->float:
    y_pred = predict(X, w_0, w_1)
    return (-2*(y-y_pred).mean()).item()

In [ ]:
#@title TRANSFORMER
class StandardScaler:
    """
    Standardize features by removing the mean and scaling to unit variance.

    For each feature (column) the transformation is:
        z = (x - mean) / std

    The scaler supports both 1D arrays (single feature) and 2D arrays
    (multiple features, one per column). The output shape always matches
    the input shape.

    Attributes
    ----------
    mean_ : np.ndarray or float
        Per-feature mean computed during `fit`.
        Scalar for 1D input, array of shape (n_features,) for 2D input.
    std_ : np.ndarray or float
        Per-feature standard deviation computed during `fit`.
        Scalar for 1D input, array of shape (n_features,) for 2D input.
    """

    def __init__(self):
        self.mean_ = None
        self.std_ = None
        self._is_1d = None  # remembers input dimensionality

    def fit(self, X):
        """
        Compute the mean and standard deviation to be used for later scaling.

        Parameters
        ----------
        X : array-like of shape (n_samples,) or (n_samples, n_features)
            The data used to compute the per-feature mean and std.

        Returns
        -------
        self : StandardScaler
            The fitted scaler (allows method chaining).
        """
        X = np.asarray(X, dtype=float)
        self._is_1d = (X.ndim == 1)

        # axis=0 reduces along samples; for 1D this just returns a scalar
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        return self

    def transform(self, X):
        """
        Standardize the data using the previously computed mean and std.

        Parameters
        ----------
        X : array-like of shape (n_samples,) or (n_samples, n_features)
            The data to be standardized. Must have the same number of
            features as the data passed to `fit`.

        Returns
        -------
        X_scaled : np.ndarray
            Standardized data with the same shape as the input.
        """
        if self.mean_ is None or self.std_ is None:
            raise RuntimeError("Scaler must be fitted before calling transform().")
        X = np.asarray(X, dtype=float)
        return (X - self.mean_) / self.std_

    def fit_transform(self, X):
        """
        Fit the scaler to X and return the standardized version of X.

        Parameters
        ----------
        X : array-like of shape (n_samples,) or (n_samples, n_features)
            The data to fit and transform.

        Returns
        -------
        X_scaled : np.ndarray
            Standardized data with the same shape as the input.
        """
        return self.fit(X).transform(X)

    def inverse(self, X_scaled):
        """
        Undo the standardization: map scaled data back to the original scale.

        Applies the inverse transformation:
            x = z * std + mean

        Parameters
        ----------
        X_scaled : array-like of shape (n_samples,) or (n_samples, n_features)
            Previously standardized data.

        Returns
        -------
        X : np.ndarray
            Data on the original scale, with the same shape as the input.
        """
        if self.mean_ is None or self.std_ is None:
            raise RuntimeError("Scaler must be fitted before calling inverse().")
        X_scaled = np.asarray(X_scaled, dtype=float)
        return X_scaled * self.std_ + self.mean_


Прежде чем запускать обучение, разберёмся, зачем мы стандартизировали признак. Посмотрите на исходные данные: рост в среднем около 170 см при разбросе порядка 9 см. То есть значения признака — это большие числа, плотно сгруппированные вокруг 170. В формулах для частных производных это приводит к дисбалансу: производная по $w_0$ зависит от средней ошибки, а в производную по $w_1$ каждая ошибка входит ещё и умноженной на $x_i \approx 170$. В результате одна компонента градиента оказывается в десятки раз больше другой, и функция потерь приобретает форму вытянутого оврага вместо симметричной чаши. На таком ландшафте градиентный спуск работает плохо: при большом шаге обучения итерации перескакивают через овраг и расходятся, при малом — сходимость вдоль оврага становится неприемлемо медленной. Подобрать рабочее значение `lr` в такой ситуации затруднительно. Стандартизация $$z = \frac{x - \mu}{\sigma}$$ устраняет эту проблему: после неё признак имеет нулевое среднее и единичную дисперсию, обе частные производные становятся сопоставимыми по величине, линии уровня функции потерь близки к концентрическим окружностям, и градиентный спуск устойчиво сходится при широком диапазоне значений шага. Поэтому стандартизация в данной задаче — не формальность, а необходимое условие сходимости. Можете попробовать запустить обучение без нее и оценить весь уровень трэша! (ха - ха)

In [ ]:
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

In [ ]:
scaler.std_, scaler.mean_

In [ ]:
sns.kdeplot(X_std)

In [ ]:
scaler.mean_, scaler.std_

## Model Init

In [ ]:
X_std

In [ ]:
Learning_Rate = 1e-2
N_ITER = 400
W_0, W_1 = np.random.randn(2)
W_0, W_1

In [ ]:
#@title Init Model Plot
xxx = np.linspace(-3, 3, 10)
yyy = W_0 + W_1*xxx
sns.scatterplot(x=X_std, y=y, color='grey', s=60, alpha=.8, edgecolor='black')
plt.title('Train Data')
sns.lineplot(x=xxx, y=yyy, color='red', label='Initial Linear model')
plt.grid()

## Train model

In [ ]:
loss_steps = []
norm_grad = []
w_history = []  # history of (W_0, W_1) for the loss surface plot

for _ in range(N_ITER):
    w_history.append((W_0, W_1))
    y_pred = predict(X_std, W_0, W_1)
    loss_ = mse_loss(y, y_pred)
    loss_steps.append(loss_)
    gradient = np.array([dl_dw_0(X_std, y, W_0, W_1), dl_dw_1(X_std, y, W_0, W_1)])
    norm_grad.append(np.linalg.norm(gradient).item())
    W_0 = W_0 - Learning_Rate * gradient[0]
    W_1 = W_1 - Learning_Rate * gradient[1]

w_history = np.array(w_history)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.lineplot(norm_grad, color='black', ax=ax[0]).set_title('Gradient norma')
sns.lineplot(loss_steps, color='green', ax=ax[1]).set_title('LOSS')
ax[0].grid()
ax[1].grid()

In [ ]:
W_0, W_1

In [ ]:
#@title Trained Model plot
xxx = np.linspace(-3, 4, 10)
yyy = W_0 + W_1*xxx
sns.scatterplot(x=X_std, y=y, color='grey', s=60, alpha=.8, edgecolor='black')
plt.title('Train Data')
sns.lineplot(x=xxx, y=yyy, color='red', label='Trained Linear model')
plt.grid()

In [ ]:
#@title GD plot
# --- Build a grid over (W_0, W_1) and compute MSE on it ---
w0_range = np.linspace(w_history[:, 0].min() - 5, w_history[:, 0].max() + 5, 100)
w1_range = np.linspace(w_history[:, 1].min() - 5, w_history[:, 1].max() + 5, 100)
W0, W1 = np.meshgrid(w0_range, w1_range)
L = np.zeros_like(W0)

for i in range(W0.shape[0]):
    for j in range(W0.shape[1]):
        L[i, j] = mse_loss(y, predict(X_std, W0[i, j], W1[i, j]))

# --- Plot: 3D surface on the left, contour map on the right ---
fig = plt.figure(figsize=(15, 6))

# Left: 3D loss surface with descent trajectory
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(W0, W1, L, cmap='viridis', alpha=0.6, edgecolor='none')
ax1.plot(w_history[:, 0], w_history[:, 1], loss_steps,
         color='red', linewidth=2, marker='o', markersize=3, label='GD trajectory')
ax1.set_xlabel("w_0")
ax1.set_ylabel("w_1")
ax1.set_zlabel("MSE")
ax1.set_title("Loss Surface L(w_0, w_1)")
ax1.legend()

# Right: contour map with descent trajectory
ax2 = fig.add_subplot(1, 2, 2)
contours = ax2.contour(W0, W1, L, levels=20, colors='black', linewidths=0.8)
ax2.clabel(contours, inline=True, fontsize=7, fmt="%.0f")
ax2.plot(w_history[:, 0], w_history[:, 1],
         color='red', linewidth=1.5, marker='o', markersize=4, label='GD trajectory')
ax2.plot(w_history[0, 0], w_history[0, 1], 'go', markersize=10, label='start')
ax2.plot(w_history[-1, 0], w_history[-1, 1], 'b*', markersize=15, label='end')
ax2.set_xlabel("w_0")
ax2.set_ylabel("w_1")
ax2.set_title("Contour Lines of L(w_0, w_1) with Descent Path")
ax2.grid(True, linestyle='--', alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

## Predict | Evaluation

In [ ]:
W_0, W_1

Мы получили $w_0 \approx 64.66$ и $w_1 \approx 8.19$. Признак у нас стандартизирован $$z = \frac{x - \mu}{\sigma}$$ где $\mu \approx 170$ см, $\sigma \approx 9$ см, поэтому коэффициенты интерпретируются в терминах $z$, а не сантиметров.

**Свободный член $w_0 \approx 64.66$.** Это предсказание модели при $z = 0$, то есть для человека среднего роста (170 см). Получается, человек со средним ростом весит по модели около 64.7 кг.

**Коэффициент наклона $w_1 \approx 8.19$.** Показывает, на сколько килограммов меняется предсказание при изменении $z$ на единицу — то есть при росте, увеличенном на одно стандартное отклонение ($\sigma \approx 9$ см). Иначе говоря, человек, который выше среднего на 9 см, весит по прогнозу примерно на 8.2 кг больше.

In [ ]:
y_pred = predict(X_std, W_0, W_1)
error = y - y_pred

mae = np.abs(error).mean().item()
mape = (np.abs(error) / y).mean().item()

print(f"MAE  : {mae:.2f} кг")
print(f"MAPE : {mape * 100:.2f} %")

Для оценки качества обученной модели вычислим две стандартные метрики регрессии.

**Средняя абсолютная ошибка (Mean Absolute Error, MAE):**

$$
\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} \left| y_i - \hat{y}_i \right|.
$$

Метрика измеряется в тех же единицах, что и целевая переменная (в нашем случае — в килограммах), и показывает, на сколько в среднем предсказание модели отклоняется от истинного значения. Полученное значение $\text{MAE} \approx 4.18$ кг означает, что в среднем прогноз массы по росту отличается от фактического веса примерно на 4 кг.

**Средняя абсолютная процентная ошибка (Mean Absolute Percentage Error, MAPE):**

$$
\text{MAPE} = \frac{1}{n} \sum_{i=1}^{n} \frac{\left| y_i - \hat{y}_i \right|}{\left| y_i \right|}.
$$

В отличие от MAE, эта метрика безразмерна и выражает относительную ошибку — какую долю от истинного значения составляет среднее отклонение. Значение $\text{MAPE} \approx 6.70\%$ показывает, что средняя ошибка прогноза составляет около 7% от реального веса. Это, по меркам индустрии, более чем отличный результат (стандартом часто считают величину около 10%).

MAE удобна для содержательной интерпретации в исходных единицах, тогда как MAPE позволяет сопоставлять качество моделей, обученных на данных разного масштаба. Полученные значения свидетельствуют о приемлемой точности линейной модели с одним признаком.

In [ ]:
#@title error plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ---------- Left: error distribution ----------
sns.kdeplot(error, color='black', ax=ax1)
ax1.axvline(0, color='red', linestyle='--', linewidth=1)
ax1.set_title("Error Distribution")
ax1.set_xlabel("error = y - y_pred, kg")
ax1.set_ylabel("Density")
ax1.grid(True, linestyle='--', alpha=0.4)

# ---------- Right: error vs true value ----------
sns.scatterplot(x=y, y=error, color='grey', s=60, alpha=0.8,
                edgecolor='black', ax=ax2)
ax2.axhline(0, color='red', linestyle='--', linewidth=1)
ax2.set_title("Error vs. True Value")
ax2.set_xlabel("y (true mass), kg")
ax2.set_ylabel("error = y - y_pred, kg")
ax2.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

Эти два графика — стандартный диагностический инструмент для регрессии, и каждый из них отвечает на свой вопрос.

**Слева — распределение остатков (residual distribution).** Плотность ошибок $e_i = y_i - \hat{y}_i$ построена с помощью ядерной оценки (KDE). Для корректно подобранной линейной модели мы ожидаем, что распределение остатков будет приближённо нормальным, симметричным относительно нуля и без тяжёлых хвостов. На полученном графике пик расположен вблизи нуля, форма близка к колоколообразной — это свидетельствует об отсутствии систематического смещения предсказаний, то есть модель в среднем не занижает и не завышает массу. Хвосты доходят примерно до $\pm 15$ кг, что согласуется со значением $\text{MAE} \approx 4.18$ кг.

**Справа — диаграмма зависимости ошибки от истинного значения целевой переменной.** Этот график позволяет проверить однородность дисперсии ошибок (свойство гомоскедастичности) и обнаружить систематические смещения на разных диапазонах $y$. В идеале точки должны равномерно располагаться вокруг горизонтальной линии $e = 0$, не образуя видимых трендов. На нашем графике хорошо заметна положительная зависимость: при малых значениях массы (40–55 кг) остатки преимущественно отрицательны (модель завышает), а при больших (75–90 кг) — положительны (модель занижает). Это типичный признак того, что линейная зависимость недостаточно хорошо описывает данные на краях распределения, и модель «слишком плоская» по сравнению с истинной связью между ростом и массой. Также наблюдается рост дисперсии ошибок с увеличением $y$ — гетероскедастичность, что характерно для биометрических данных: у людей с большей массой разброс веса при фиксированном росте естественно больше.

Совокупно эти графики показывают, что модель в целом несмещённая и работает приемлемо, но имеет ограничения на краях диапазона — улучшить её можно добавлением дополнительных признаков или нелинейных преобразований.

# Task

С помощью нашей обученной модели сделайте прогноз массы тела для этих значений роста:
1) 100 sm
2) 170 sm
3) 210 sm
5) 500 sm
6) -10 sm

Ничего не смущает? Если нет то все норм! Если да, то тут кк раз проявляется свойство линейной модели, являющееся как преимуществом, так и недостатком... как посмотреть. Подумайте, о чем идет речь!

In [ ]:
W_0, W_1

In [ ]:
sample = 170
sampl_std = scaler.transform(sample)
sampl_std

In [ ]:
predict(sampl_std, W_0, W_1)



---



# Линейная модель в матричном виде

В пердыдущей главе мы работали с моделью покомпонентно: отдельно вычисляли частную производную по $w_1$, отдельно — по $w_0$. Такой подход хорош для понимания, но имеет очевидный недостаток: при добавлении новых признаков нам пришлось бы вручную выписывать формулы для каждого веса в отдельности, что такое себе... На практике линейные модели работают с десятками и сотнями признаков, и индивидуальный учёт каждого веса становится непрактичным. Матричная запись позволяет описать всю модель — предсказание, функцию потерь и градиент — единообразно, не зависящим от размерности образом, и обновлять все веса одной формулой.

---

**Подготовка данных.** Чтобы не хранить и не обновлять свободный член $w_0$ отдельно от остальных весов, к матрице признаков добавляют слева столбец из единиц:

$$
X = \begin{pmatrix}
1 & x_{1,1} & x_{1,2} & \cdots & x_{1,d} \\
1 & x_{2,1} & x_{2,2} & \cdots & x_{2,d} \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & x_{n,1} & x_{n,2} & \cdots & x_{n,d}
\end{pmatrix} \in \mathbb{R}^{n \times (d+1)}.
$$

Этот фиктивный единичный столбец берёт на себя роль свободного члена: соответствующий ему вес становится обычным элементом вектора параметров и обновляется по той же формуле, что и остальные. В нашем случае с одним признаком (рост) матрица $X$ имеет размер $n \times 2$: первый столбец — единицы, второй — стандартизированные значения роста.

Вектор весов записывается как

$$
w = \begin{pmatrix} w_0 \\ w_1 \\ \vdots \\ w_d \end{pmatrix} \in \mathbb{R}^{d+1},
$$

где $w_0$ — свободный член (тот самый, что раньше был отдельной переменной), а $w_1, \ldots, w_d$ — веса при признаках. Целевая переменная — вектор $y \in \mathbb{R}^n$.

---

**Линейная модель в матричном виде.** Предсказание модели для всех $n$ наблюдений одновременно записывается одним матрично-векторным произведением:

$$
\hat{y} = X w \in \mathbb{R}^n.
$$

Для отдельного объекта $i$-е предсказание имеет вид $\hat{y}_i = w_0 + w_1 x_{i,1} + \ldots + w_d x_{i,d}$ — то же самое, что и раньше, но записано компактно.

---

**Loss.** Среднеквадратичная ошибка в матричной форме:

$$
L(w) = \frac{1}{n} \| y - Xw \|^2 = \frac{1}{n} (y - Xw)^\top (y - Xw),
$$

где $\| \cdot \|$ — евклидова норма вектора. Внутри нормы стоит вектор остатков $e = y - Xw$ длины $n$, и его квадрат нормы — это в точности сумма квадратов невязок по всем наблюдениям.

---

**Градиент функции потерь.** Самое главное достоинство матричной записи — компактная формула градиента, единая для всех весов сразу:

$$
\nabla_w L(w) = -\frac{2}{n} X^\top (y - Xw) \in \mathbb{R}^{d+1}.
$$

Это вектор той же размерности, что и $w$: его $k$-я компонента — это частная производная $\partial L / \partial w_k$. Если расписать формулу покомпонентно для случая одного признака с единичным столбцом, получатся ровно те же выражения, что мы выводили вручную в предыдущей главе — но теперь они получены одним движением и без разделения на «производную по смещению» и «производную по наклону».

---

**Шаг градиентного спуска** в матричном виде также превращается в одну строку:

$$
w \leftarrow w - \eta \cdot \nabla_w L(w) = w + \frac{2 \eta}{n} X^\top (y - Xw),
$$

где $\eta$ — скорость схождения (LR). Эта формула одинаково работает для любого числа признаков — что для одного, что для тысячи. Ниже обучим модель в матричной записи и убедимся, что результат совпадает с тем, что мы получили в покомпонентной реализации.


После обучения коэффициенты должны совпасть с теми, что мы получили в скалярной реализации — $w_0 \approx 64.66$, $w_1 \approx 8.19$, — что подтверждает эквивалентность подходов. Преимущество матричной записи в том, что эти три функции (`predict`, `mse_loss`, `grad`) теперь работают для модели с любым числом признаков без единого изменения — достаточно просто подать матрицу $X$ нужной размерности.

# Task

Выполним предложенную задачу и оценим результат.

In [ ]:
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

In [ ]:
scaler.mean_, scaler.std_

In [ ]:
X_std[:4], y[:2]

In [ ]:
X_mod = np.array([np.ones(X_std.shape), X_std]).T

In [ ]:
X_mod

In [ ]:
def predict(X: np.ndarray, W: np.ndarray)->np.ndarray:
  return X @ W

def loss(X, y, W):
  error = y - X @ W
  return (error.T @ error) / X.shape[0]

def gradient(X, y, W):
  error = y - X @ W
  return (-1/X.shape[0]) * X.T @ error


In [ ]:
Learning_Rate = 1e-2
N_ITER = 500
W = np.random.randn(2)
W

In [ ]:
loss_steps = []
norm_grad = []
w_history = []  # history of (W_0, W_1) for the loss surface plot

for _ in range(N_ITER):
    w_history.append(W)
    y_pred = predict(X_mod, W)
    loss_ = loss(X_mod, y, W)

    loss_steps.append(loss_)
    grad = gradient(X_mod, y, W)
    norm_grad.append(np.linalg.norm(grad).item())
    W = W - Learning_Rate * grad

w_history = np.array(w_history)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.lineplot(norm_grad, color='black', ax=ax[0]).set_title('Gradient norma')
sns.lineplot(loss_steps, color='green', ax=ax[1]).set_title('LOSS')
ax[0].grid()
ax[1].grid()

In [ ]:
W

In [ ]:
fig, ax = plt.subplots(2, figsize=(10, 6))
sns.lineplot(w_history[:, 0], ax=ax[0], color='black').set_title('bias history $\omega_0$')
sns.lineplot(w_history[:, 1], ax=ax[1], color='black').set_title('slope history $\omega_1$')
ax[0].grid()
ax[1].grid()

In [ ]:
#@title Trained Model plot
xxx = np.linspace(-3, 4, 10)
yyy = W[0] + W[1]*xxx
sns.scatterplot(x=X_mod[:, -1], y=y, color='grey', s=60, alpha=.8, edgecolor='black')
plt.title('Train Data')
sns.lineplot(x=xxx, y=yyy, color='red', label='Trained Linear model')
plt.grid()

In [ ]:
y_pred = predict(X_mod, W)
error = y - y_pred

mae = np.abs(error).mean().item()
mape = (np.abs(error) / y).mean().item()

print(f"MAE  : {mae:.2f} кг")
print(f"MAPE : {mape * 100:.2f} %")

In [ ]:
W

# Task ***

# Задача: предсказание вероятности поступления

Перед нами популярный датасет `Graduate Admissions`. Он содержит информацию о кандидатах, поступающих в магистратуру, и используется для предсказания вероятности их зачисления. В нём представлены такие характеристики, как баллы GRE и TOEFL, рейтинг университета, оценки мотивационного письма и рекомендательных писем, средний балл (CGPA), а также наличие исследовательского опыта. Целевой признак — вероятность поступления, выраженная числом от 0 до 1.

## Постановка задачи

Используя матричную реализацию линейной регрессии из предыдущего раздела, обучите модель предсказывать вероятность поступления $y \in [0, 1]$ по набору признаков кандидата. Модель имеет вид:

$$
\hat{y} = X w, \qquad w \in \mathbb{R}^{d+1},
$$

где $X$ — матрица признаков с дописанным слева единичным столбцом для свободного члена, а $w$ — вектор весов, содержащий смещение $w_0$ и коэффициенты при $d$ признаках.

## Что нужно сделать

**1. Подготовка данных.** Разделите датасет на матрицу признаков $X$ и вектор целевых значений $y$. Стандартизируйте признаки с помощью реализованного ранее класса `StandardScaler` — это критически важно, поскольку признаки имеют существенно разные масштабы (GRE — сотни, CGPA — единицы, бинарный признак исследовательского опыта — нули и единицы), и без стандартизации градиентный спуск не сойдётся при разумном шаге обучения. Разделите выборку на обучающую и тестовую части в пропорции примерно 80/20. Допишите к матрице $X$ слева столбец из единиц для учёта свободного члена.

**2. Обучение модели.** Инициализируйте вектор весов $w$ случайными значениями. Реализуйте цикл градиентного спуска, на каждой итерации обновляющий веса по формуле

$$
w \leftarrow w - \eta \cdot \nabla_w L(w), \qquad \nabla_w L(w) = -\frac{2}{n} X^\top (y - Xw).
$$

Подберите скорость обучения $\eta$ и число эпох так, чтобы функция потерь сходилась к стабильному значению. На каждой эпохе сохраняйте значение MSE — это понадобится для построения кривой обучения.

**3. Оценка качества.** Вычислите на тестовой выборке метрики MAE и MAPE и сравните с результатами на обучающей выборке. Постройте три диагностических графика: кривую обучения (зависимость MSE от номера эпохи), распределение остатков и диаграмму «ошибка против истинного значения».

**4. Интерпретация коэффициентов.** Поскольку признаки стандартизированы, абсолютные значения весов $w_1, \ldots, w_d$ напрямую сопоставимы между собой и характеризуют относительную важность каждого признака для предсказания. Ранжируйте признаки по модулю коэффициента и обсудите, какие из них вносят наибольший вклад в вероятность поступления, а какие практически не влияют. Прокомментируйте знаки коэффициентов с содержательной точки зрения: согласуются ли они со здравым смыслом?

## Контрольные вопросы

Что произойдёт, если пропустить этап стандартизации признаков и запустить обучение на исходных данных? Почему столбец из единиц позволяет обойтись без отдельной обработки свободного члена? Можно ли по полученной модели судить о причинно-следственной связи между признаками и вероятностью поступления, или речь идёт лишь о корреляционной зависимости?

In [ ]:
#@title load data
!wget 'https://www.dropbox.com/scl/fi/voe83n4r51yhuppim5jjp/AMD_dataset.csv?rlkey=bzk5qeuh67klg61e9i85n7oy9&dl=0' -O AMD_dataset.csv

rename_dict = {
    "GRE Score": "gre_score",
    "TOEFL Score": "toefl_score",
    "University Rating": "university_rating",
    "SOP": "sop",
    "LOR": "lor",
    "CGPA": "cgpa",
    "Research": "research",
    "Chance of Admit ": "chance_of_admit"
}

df = pd.read_csv('/content/AMD_dataset.csv').drop('Serial No.', axis=1)
df = df.rename(rename_dict, axis=1)

In [ ]:
df

In [ ]:
df.columns

In [ ]:
X = df.drop(['chance_of_admit'], axis=1).values
y = df.chance_of_admit.values

In [ ]:
X[:5], y[:10]

In [ ]:
sns.kdeplot(y, color='black')